
# YOLO26 + Semantic Prototype Feasibility Test on RDD2022-China

This Colab notebook is designed for a **single NVIDIA T4**.

It keeps the uploaded YOLO26s baseline workflow and adds a safe first experiment for semantic embeddings:

1. Train or reuse the normal YOLO26s detector.
2. Encode road-damage text descriptions with a CLIP text encoder.
3. Encode ground-truth road-damage crops with the matching CLIP image encoder.
4. Compare:
   - class-name prompts,
   - discriminative prompt ensembles,
   - train-set visual prototypes,
   - fused text + visual prototypes.
5. Reclassify YOLO-predicted boxes using the best semantic prototype setting.
6. Measure whether semantic knowledge reduces class confusion, especially:
   - longitudinal vs transverse cracks,
   - pothole vs repair,
   - alligator crack vs repair.

## Important interpretation

This is a **feasibility and ablation notebook**, not yet a full replacement of the YOLO26 classification head.

It answers the first research question:

> Do text-derived semantic prototypes contain useful class-separation information for this road-damage dataset?

If the answer is yes, the next step is to integrate the learned projection or semantic regularization directly into the YOLO26 classification head.


## 1. Install dependencies

In [ ]:

!pip -q install -U ultralytics open_clip_torch scikit-learn pandas pillow pyyaml


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
dask-cudf-cu1

In [ ]:

import os, gc, json, math, random, shutil, zipfile, yaml
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, balanced_accuracy_score
from ultralytics import YOLO

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f"GPU memory free/total: {free/1e9:.2f}/{total/1e9:.2f} GB")

assert torch.cuda.is_available(), "Select a Colab T4 runtime: Runtime > Change runtime type > T4 GPU"
DEVICE = "cuda"
torch.backends.cudnn.benchmark = True


## 2. Mount Drive and configure paths

In [ ]:

from google.colab import drive
drive.mount("/content/drive")


In [ ]:

# ---------------- USER CONFIG ----------------
DRIVE_ZIP = Path("/content/drive/MyDrive/Research/Dataset/China_Subset.zip")
LOCAL_ROOT = Path("/content/China_Subset")

OUT_DIR = Path("/content/drive/MyDrive/Research/runs_rdd2022/yolo26s_semantic_test")
BASE_MODEL = "yolo26s"
EPOCHS = 100
IMGSZ = 640

# T4-safe settings
TRAIN_BATCH = 16
VAL_BATCH = 8
CLIP_BATCH = 64
MAX_TRAIN_CROPS_PER_CLASS = 1500  # Reduce to 800 if Drive/data loading is slow
MAX_VAL_CROPS_PER_CLASS = 600
MAX_TEST_CROPS_PER_CLASS = 1000
NUM_WORKERS = 2
SEED = 0
# ---------------------------------------------

CLASS_NAMES = [
    "longitudinal crack",
    "transverse crack",
    "alligator crack",
    "pothole",
    "repair",
]
NUM_CLASSES = len(CLASS_NAMES)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

OUT_DIR.mkdir(parents=True, exist_ok=True)

if not LOCAL_ROOT.exists():
    assert DRIVE_ZIP.exists(), f"Dataset zip not found: {DRIVE_ZIP}"
    local_zip = Path("/content/China_Subset.zip")
    shutil.copy2(DRIVE_ZIP, local_zip)
    with zipfile.ZipFile(local_zip) as zf:
        zf.extractall("/content")
    print("Dataset extracted.")

yaml_path = LOCAL_ROOT / "dataset.yaml"
cfg = {
    "path": str(LOCAL_ROOT),
    "train": "train_set/images",
    "val": "valid_set/images",
    "test": "test_set/images",
    "nc": NUM_CLASSES,
    "names": CLASS_NAMES,
}
with open(yaml_path, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

for split in ["train_set", "valid_set", "test_set"]:
    image_dir = LOCAL_ROOT / split / "images"
    label_dir = LOCAL_ROOT / split / "labels"
    n_images = len([p for p in image_dir.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])
    n_labels = len(list(label_dir.glob("*.txt")))
    print(f"{split:10s}: {n_images:5d} images | {n_labels:5d} labels")

print("Dataset YAML:", yaml_path)
print(yaml.safe_load(open(yaml_path)))



## 3. Train or reuse the YOLO26s baseline

This preserves the original detector setup. Semantic experiments are performed afterward, so a failure in the semantic section will not invalidate the detector checkpoint.


In [ ]:

CKPT = OUT_DIR / "train" / "weights" / "best.pt"

if CKPT.exists():
    print("Reusing existing checkpoint:", CKPT)
else:
    gc.collect()
    torch.cuda.empty_cache()

    YOLO(BASE_MODEL).train(
        data=str(yaml_path),
        project=str(OUT_DIR),
        name="train",
        exist_ok=True,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=TRAIN_BATCH,
        device=0,
        workers=NUM_WORKERS,
        optimizer="SGD",
        lr0=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=3,
        mosaic=1.0,
        patience=0,
        cache=False,
        amp=True,
        plots=False,
        seed=SEED,
    )

assert CKPT.exists(), f"Training did not produce {CKPT}"
print("Checkpoint:", CKPT)


## 4. Official YOLO26 test metrics

In [ ]:

gc.collect()
torch.cuda.empty_cache()

detector = YOLO(str(CKPT))
metrics = detector.val(
    data=str(yaml_path),
    split="test",
    imgsz=IMGSZ,
    batch=VAL_BATCH,
    device=0,
    verbose=False,
)

rows = [{
    "class": "ALL",
    "mAP50": float(metrics.box.map50),
    "mAP50_95": float(metrics.box.map),
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
}]

for i, name in metrics.names.items():
    rows.append({
        "class": name,
        "mAP50": float(metrics.box.ap50[i]),
        "mAP50_95": float(metrics.box.maps[i]),
        "precision": np.nan,
        "recall": np.nan,
    })

official_df = pd.DataFrame(rows)
display(official_df.round(4))
official_df.to_csv(OUT_DIR / "official_yolo_metrics.csv", index=False)

del detector, metrics
gc.collect()
torch.cuda.empty_cache()



## 5. Build ground-truth crop records

Each labeled road-damage box becomes one crop. A small padding factor is added because tight boxes can remove useful surrounding road texture.

The crop records are cached as CSV; image pixels are loaded only when needed.


In [ ]:

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
CROP_PADDING = 0.10

def find_image_by_stem(image_dir: Path, stem: str):
    for ext in IMAGE_EXTENSIONS:
        p = image_dir / f"{stem}{ext}"
        if p.exists():
            return p
        p = image_dir / f"{stem}{ext.upper()}"
        if p.exists():
            return p
    matches = list(image_dir.glob(f"{stem}.*"))
    return matches[0] if matches else None

def build_crop_records(split: str) -> pd.DataFrame:
    image_dir = LOCAL_ROOT / split / "images"
    label_dir = LOCAL_ROOT / split / "labels"
    rows = []

    for label_path in sorted(label_dir.glob("*.txt")):
        image_path = find_image_by_stem(image_dir, label_path.stem)
        if image_path is None:
            continue

        with Image.open(image_path) as im:
            width, height = im.size

        text = label_path.read_text().strip()
        if not text:
            continue

        for obj_idx, line in enumerate(text.splitlines()):
            parts = line.split()
            if len(parts) < 5:
                continue

            cls_id = int(float(parts[0]))
            xc, yc, bw, bh = map(float, parts[1:5])

            x1 = (xc - bw / 2) * width
            y1 = (yc - bh / 2) * height
            x2 = (xc + bw / 2) * width
            y2 = (yc + bh / 2) * height

            pad_x = (x2 - x1) * CROP_PADDING
            pad_y = (y2 - y1) * CROP_PADDING

            x1 = max(0, int(math.floor(x1 - pad_x)))
            y1 = max(0, int(math.floor(y1 - pad_y)))
            x2 = min(width, int(math.ceil(x2 + pad_x)))
            y2 = min(height, int(math.ceil(y2 + pad_y)))

            if x2 <= x1 or y2 <= y1 or not (0 <= cls_id < NUM_CLASSES):
                continue

            rows.append({
                "split": split,
                "image_path": str(image_path),
                "stem": image_path.stem,
                "object_index": obj_idx,
                "class_id": cls_id,
                "class_name": CLASS_NAMES[cls_id],
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "width": x2 - x1,
                "height": y2 - y1,
                "aspect_ratio": max((x2 - x1) / max(y2 - y1, 1), (y2 - y1) / max(x2 - x1, 1)),
            })

    return pd.DataFrame(rows)

records = {}
limits = {
    "train_set": MAX_TRAIN_CROPS_PER_CLASS,
    "valid_set": MAX_VAL_CROPS_PER_CLASS,
    "test_set": MAX_TEST_CROPS_PER_CLASS,
}

for split, limit in limits.items():
    df = build_crop_records(split)
    sampled = []
    for cls_id in range(NUM_CLASSES):
        part = df[df.class_id == cls_id]
        if len(part) > limit:
            part = part.sample(limit, random_state=SEED)
        sampled.append(part)
    df = pd.concat(sampled, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
    records[split] = df
    df.to_csv(OUT_DIR / f"{split}_crop_records.csv", index=False)
    print(f"\n{split}: {len(df)} crops")
    print(df.class_name.value_counts().reindex(CLASS_NAMES).fillna(0).astype(int).to_string())



## 6. Load CLIP and define prompt ensembles

`ViT-B-32` is used because it is reliable and T4-friendly. The notebook uses the same model for both image and text embeddings, so the vectors are directly comparable.

The enriched prompts explicitly describe orientation, texture, compactness, and patch appearance.


In [ ]:

import open_clip

CLIP_MODEL_NAME = "ViT-B-32"
CLIP_PRETRAINED = "laion2b_s34b_b79k"

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    CLIP_MODEL_NAME,
    pretrained=CLIP_PRETRAINED,
    device=DEVICE,
)
clip_tokenizer = open_clip.get_tokenizer(CLIP_MODEL_NAME)
clip_model.eval()

for p in clip_model.parameters():
    p.requires_grad_(False)

print("Loaded:", CLIP_MODEL_NAME, CLIP_PRETRAINED)


In [ ]:

NAME_PROMPTS = {
    0: ["a photo of a longitudinal road crack"],
    1: ["a photo of a transverse road crack"],
    2: ["a photo of an alligator road crack"],
    3: ["a photo of a pothole"],
    4: ["a photo of a repaired road surface"],
}

RICH_PROMPTS = {
    0: [
        "a long narrow pavement crack running parallel to the traffic direction",
        "an elongated longitudinal road crack aligned with the road",
        "a mostly road-aligned linear crack on asphalt pavement",
        "a thin continuous crack extending along the road",
    ],
    1: [
        "a long pavement crack running across the traffic direction",
        "an elongated transverse road crack crossing the road",
        "a mostly cross-road linear crack on asphalt pavement",
        "a thin continuous crack approximately perpendicular to the road direction",
    ],
    2: [
        "an interconnected network of polygon-shaped pavement cracks",
        "alligator cracking with branching web-like crack segments",
        "a dense cracked asphalt region resembling reptile skin",
        "many connected short cracks forming irregular cells",
    ],
    3: [
        "an irregular depressed cavity with missing asphalt on a road",
        "a compact pothole with broken pavement and a dark interior",
        "a localized road hole with rough irregular boundaries",
        "a damaged pavement cavity deeper than the surrounding road",
    ],
    4: [
        "a repaired asphalt patch with a different texture from the surrounding road",
        "a road repair region covered with newer or darker pavement material",
        "an irregular pavement patch covering previous road damage",
        "a repaired road surface area with visible patch boundaries",
    ],
}

@torch.no_grad()
def encode_prompt_dictionary(prompt_dict):
    class_vectors = []
    for cls_id in range(NUM_CLASSES):
        texts = prompt_dict[cls_id]
        tokens = clip_tokenizer(texts).to(DEVICE)
        vecs = clip_model.encode_text(tokens)
        vecs = F.normalize(vecs.float(), dim=-1)
        prototype = F.normalize(vecs.mean(dim=0, keepdim=True), dim=-1)
        class_vectors.append(prototype)
    return torch.cat(class_vectors, dim=0)

name_text_prototypes = encode_prompt_dictionary(NAME_PROMPTS)
rich_text_prototypes = encode_prompt_dictionary(RICH_PROMPTS)

torch.save(name_text_prototypes.cpu(), OUT_DIR / "name_text_prototypes.pt")
torch.save(rich_text_prototypes.cpu(), OUT_DIR / "rich_text_prototypes.pt")

print("Text prototype shape:", tuple(rich_text_prototypes.shape))


## 7. Encode crop images and cache embeddings

In [ ]:

from torch.utils.data import Dataset, DataLoader

class CropDataset(Dataset):
    def __init__(self, dataframe, preprocess):
        self.df = dataframe.reset_index(drop=True)
        self.preprocess = preprocess

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        with Image.open(row.image_path) as im:
            im = im.convert("RGB")
            crop = im.crop((int(row.x1), int(row.y1), int(row.x2), int(row.y2)))
        return self.preprocess(crop), int(row.class_id), index

@torch.no_grad()
def encode_crop_dataframe(df, cache_path):
    cache_path = Path(cache_path)
    if cache_path.exists():
        obj = torch.load(cache_path, map_location="cpu")
        if len(obj["labels"]) == len(df):
            print("Loaded cache:", cache_path)
            return obj["embeddings"], obj["labels"]

    loader = DataLoader(
        CropDataset(df, clip_preprocess),
        batch_size=CLIP_BATCH,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    all_embeddings, all_labels = [], []
    for images, labels, _ in loader:
        images = images.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            emb = clip_model.encode_image(images)
        emb = F.normalize(emb.float(), dim=-1).cpu()
        all_embeddings.append(emb)
        all_labels.append(labels.long().cpu())

    embeddings = torch.cat(all_embeddings)
    labels = torch.cat(all_labels)

    torch.save({"embeddings": embeddings, "labels": labels}, cache_path)
    print("Saved cache:", cache_path)
    return embeddings, labels

embedding_data = {}
for split in ["train_set", "valid_set", "test_set"]:
    embedding_data[split] = encode_crop_dataframe(
        records[split],
        OUT_DIR / f"{split}_clip_embeddings.pt",
    )
    print(split, embedding_data[split][0].shape)



## 8. Build train-set visual prototypes

A visual prototype is the normalized mean CLIP image embedding of one class:

\[
v_c = \operatorname{normalize}\left(\frac{1}{N_c}\sum_i x_i\right)
\]

This gives a dataset-specific representation. Text and visual prototypes are then fused.


In [ ]:

def build_visual_prototypes(embeddings, labels, num_classes):
    prototypes = []
    for cls_id in range(num_classes):
        cls_embeddings = embeddings[labels == cls_id]
        assert len(cls_embeddings) > 0, f"No samples for class {cls_id}"
        proto = F.normalize(cls_embeddings.float().mean(dim=0, keepdim=True), dim=-1)
        prototypes.append(proto)
    return torch.cat(prototypes, dim=0)

train_embeddings, train_labels = embedding_data["train_set"]
visual_prototypes = build_visual_prototypes(train_embeddings, train_labels, NUM_CLASSES)
torch.save(visual_prototypes, OUT_DIR / "visual_prototypes.pt")

print("Visual prototype shape:", tuple(visual_prototypes.shape))



## 9. Evaluate semantic prototype classifiers

Four configurations are compared:

1. class-name text prompts;
2. rich text prompts;
3. train-set visual prototypes;
4. rich text + visual fusion.

For fusion:

\[
p_c(\alpha)=\operatorname{normalize}\left(\alpha v_c +(1-\alpha)t_c\right)
\]

where \(\alpha=0\) is text-only and \(\alpha=1\) is visual-only.


In [ ]:

def classify_embeddings(embeddings, prototypes, temperature=0.01):
    embeddings = F.normalize(embeddings.float(), dim=-1)
    prototypes = F.normalize(prototypes.float(), dim=-1)
    logits = embeddings @ prototypes.T / temperature
    probs = logits.softmax(dim=1)
    pred = probs.argmax(dim=1)
    confidence = probs.max(dim=1).values
    return pred, confidence, probs

def evaluate_prototypes(name, embeddings, labels, prototypes):
    pred, confidence, probs = classify_embeddings(embeddings, prototypes)
    result = {
        "method": name,
        "accuracy": accuracy_score(labels.numpy(), pred.numpy()),
        "balanced_accuracy": balanced_accuracy_score(labels.numpy(), pred.numpy()),
        "mean_confidence": confidence.mean().item(),
    }
    return result, pred, probs

val_embeddings, val_labels = embedding_data["valid_set"]
test_embeddings, test_labels = embedding_data["test_set"]

baseline_results = []
for name, proto in [
    ("class-name text", name_text_prototypes.cpu()),
    ("rich text", rich_text_prototypes.cpu()),
    ("visual prototype", visual_prototypes.cpu()),
]:
    row, _, _ = evaluate_prototypes(name, val_embeddings, val_labels, proto)
    baseline_results.append(row)

display(pd.DataFrame(baseline_results).round(4))


In [ ]:

alpha_rows = []

for alpha in np.linspace(0.0, 1.0, 21):
    fused = F.normalize(
        alpha * visual_prototypes.float()
        + (1.0 - alpha) * rich_text_prototypes.cpu().float(),
        dim=-1,
    )
    row, _, _ = evaluate_prototypes(
        f"fusion alpha={alpha:.2f}",
        val_embeddings,
        val_labels,
        fused,
    )
    row["alpha"] = alpha
    alpha_rows.append(row)

alpha_df = pd.DataFrame(alpha_rows)
best_row = alpha_df.sort_values(
    ["balanced_accuracy", "accuracy"],
    ascending=False,
).iloc[0]

BEST_ALPHA = float(best_row.alpha)
best_fused_prototypes = F.normalize(
    BEST_ALPHA * visual_prototypes.float()
    + (1.0 - BEST_ALPHA) * rich_text_prototypes.cpu().float(),
    dim=-1,
)

print("Best validation alpha:", BEST_ALPHA)
display(alpha_df.round(4))
alpha_df.to_csv(OUT_DIR / "alpha_sweep_validation.csv", index=False)

plt.figure(figsize=(7, 4))
plt.plot(alpha_df.alpha, alpha_df.accuracy, marker="o", label="Accuracy")
plt.plot(alpha_df.alpha, alpha_df.balanced_accuracy, marker="o", label="Balanced accuracy")
plt.xlabel("Visual-prototype weight alpha")
plt.ylabel("Validation score")
plt.title("Text–visual prototype fusion")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


In [ ]:

test_methods = {
    "class-name text": name_text_prototypes.cpu(),
    "rich text": rich_text_prototypes.cpu(),
    "visual prototype": visual_prototypes.cpu(),
    f"fused alpha={BEST_ALPHA:.2f}": best_fused_prototypes.cpu(),
}

test_rows = []
test_predictions = {}

for name, proto in test_methods.items():
    row, pred, probs = evaluate_prototypes(name, test_embeddings, test_labels, proto)
    test_rows.append(row)
    test_predictions[name] = pred

test_result_df = pd.DataFrame(test_rows).sort_values("balanced_accuracy", ascending=False)
display(test_result_df.round(4))
test_result_df.to_csv(OUT_DIR / "semantic_crop_test_results.csv", index=False)


## 10. Confusion matrices and per-class report

In [ ]:

def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

    fig, ax = plt.subplots(figsize=(8, 7))
    image = ax.imshow(cm_norm, vmin=0, vmax=1)
    ax.set_xticks(range(NUM_CLASSES), CLASS_NAMES, rotation=45, ha="right")
    ax.set_yticks(range(NUM_CLASSES), CLASS_NAMES)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title(title)

    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, f"{cm[i, j]}\n{cm_norm[i, j]:.2f}", ha="center", va="center", fontsize=8)

    fig.colorbar(image, ax=ax, label="Row-normalized fraction")
    plt.tight_layout()
    plt.show()
    return cm

best_method_name = f"fused alpha={BEST_ALPHA:.2f}"
best_pred = test_predictions[best_method_name]

cm = plot_confusion(
    test_labels.numpy(),
    best_pred.numpy(),
    f"Semantic crop classification — {best_method_name}",
)

report = classification_report(
    test_labels.numpy(),
    best_pred.numpy(),
    labels=list(range(NUM_CLASSES)),
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)

report_df = pd.DataFrame(report).T
display(report_df.round(4))
report_df.to_csv(OUT_DIR / "semantic_crop_classification_report.csv")
pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
    OUT_DIR / "semantic_crop_confusion_matrix.csv"
)



## 11. Optional lightweight alignment adapter

Raw CLIP may not understand road-crack orientation well. This section learns a small linear projection using only cached crop embeddings.

The CLIP backbone remains frozen. Only a small \(d \times d\) residual adapter is trained:

\[
z = \operatorname{normalize}(x + \gamma A(x))
\]

This is much cheaper than retraining YOLO and is safe on a T4.


In [ ]:

class ResidualAdapter(torch.nn.Module):
    def __init__(self, dim, bottleneck=128):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(dim, bottleneck, bias=False),
            torch.nn.ReLU(inplace=True),
            torch.nn.Linear(bottleneck, dim, bias=False),
        )
        self.gamma = torch.nn.Parameter(torch.tensor(0.1))

    def forward(self, x):
        return F.normalize(x + self.gamma * self.net(x), dim=-1)

adapter = ResidualAdapter(train_embeddings.shape[1], bottleneck=128).to(DEVICE)
optimizer = torch.optim.AdamW(adapter.parameters(), lr=1e-3, weight_decay=1e-4)

adapter_epochs = 20
adapter_batch = 512
prototype_target = best_fused_prototypes.to(DEVICE)

train_x = train_embeddings.float()
train_y = train_labels.long()

best_adapter_state = None
best_adapter_balanced_acc = -1.0

for epoch in range(adapter_epochs):
    adapter.train()
    order = torch.randperm(len(train_x))
    losses = []

    for start in range(0, len(order), adapter_batch):
        idx = order[start:start + adapter_batch]
        x = train_x[idx].to(DEVICE)
        y = train_y[idx].to(DEVICE)

        z = adapter(x)
        logits = z @ prototype_target.T / 0.03
        loss = F.cross_entropy(logits, y)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    adapter.eval()
    with torch.no_grad():
        val_z = []
        for start in range(0, len(val_embeddings), adapter_batch):
            val_z.append(adapter(val_embeddings[start:start+adapter_batch].to(DEVICE)).cpu())
        val_z = torch.cat(val_z)
        val_pred, _, _ = classify_embeddings(val_z, prototype_target.cpu(), temperature=0.03)
        val_bal_acc = balanced_accuracy_score(val_labels.numpy(), val_pred.numpy())

    if val_bal_acc > best_adapter_balanced_acc:
        best_adapter_balanced_acc = val_bal_acc
        best_adapter_state = {k: v.detach().cpu().clone() for k, v in adapter.state_dict().items()}

    print(
        f"epoch {epoch+1:02d}/{adapter_epochs} | "
        f"loss={np.mean(losses):.4f} | val balanced acc={val_bal_acc:.4f}"
    )

adapter.load_state_dict(best_adapter_state)
torch.save(adapter.state_dict(), OUT_DIR / "best_semantic_adapter.pt")
print("Best validation balanced accuracy:", best_adapter_balanced_acc)


In [ ]:

adapter.eval()
with torch.no_grad():
    adapted_test = []
    for start in range(0, len(test_embeddings), 512):
        adapted_test.append(adapter(test_embeddings[start:start+512].to(DEVICE)).cpu())
    adapted_test = torch.cat(adapted_test)

adapter_pred, adapter_conf, adapter_probs = classify_embeddings(
    adapted_test,
    best_fused_prototypes.cpu(),
    temperature=0.03,
)

adapter_summary = pd.DataFrame([{
    "method": f"fused + residual adapter alpha={BEST_ALPHA:.2f}",
    "accuracy": accuracy_score(test_labels.numpy(), adapter_pred.numpy()),
    "balanced_accuracy": balanced_accuracy_score(test_labels.numpy(), adapter_pred.numpy()),
    "mean_confidence": adapter_conf.mean().item(),
}])

display(adapter_summary.round(4))
adapter_summary.to_csv(OUT_DIR / "semantic_adapter_test_result.csv", index=False)

plot_confusion(
    test_labels.numpy(),
    adapter_pred.numpy(),
    "Semantic crop classification — fused prototypes + adapter",
)



## 12. Generate YOLO predictions once for semantic reclassification

This section uses YOLO26 for localization. Every detected box is cropped and classified by the semantic prototype model.

The notebook then matches predicted boxes with ground truth at IoU ≥ 0.5 and compares:

- original YOLO class correctness;
- semantic reclassified correctness;
- semantic adapter correctness.

This isolates whether semantic embeddings improve the **classification part** while holding localization fixed.


In [ ]:

@torch.no_grad()
def collect_yolo_predictions(split="test_set", conf=0.05, chunk_size=32):
    image_dir = LOCAL_ROOT / split / "images"
    image_paths = sorted([
        p for p in image_dir.glob("*")
        if p.suffix.lower() in IMAGE_EXTENSIONS
    ])

    model = YOLO(str(CKPT))
    rows = []

    for start in range(0, len(image_paths), chunk_size):
        batch_paths = image_paths[start:start + chunk_size]
        results = model.predict(
            [str(p) for p in batch_paths],
            imgsz=IMGSZ,
            conf=conf,
            device=0,
            batch=4,
            stream=True,
            verbose=False,
        )

        for input_path, result in zip(batch_paths, results):
            boxes = result.boxes
            if boxes is None or len(boxes) == 0:
                continue

            xyxy = boxes.xyxy.cpu().numpy()
            cls = boxes.cls.cpu().numpy().astype(int)
            confs = boxes.conf.cpu().numpy()

            for i in range(len(xyxy)):
                rows.append({
                    "image_path": str(input_path),
                    "stem": input_path.stem,
                    "pred_index": i,
                    "yolo_class": int(cls[i]),
                    "yolo_conf": float(confs[i]),
                    "x1": float(xyxy[i, 0]),
                    "y1": float(xyxy[i, 1]),
                    "x2": float(xyxy[i, 2]),
                    "y2": float(xyxy[i, 3]),
                })

        torch.cuda.empty_cache()

    del model
    gc.collect()
    torch.cuda.empty_cache()
    return pd.DataFrame(rows)

prediction_csv = OUT_DIR / "yolo_test_predictions_low_conf.csv"
if prediction_csv.exists():
    yolo_pred_df = pd.read_csv(prediction_csv)
    print("Loaded:", prediction_csv)
else:
    yolo_pred_df = collect_yolo_predictions("test_set", conf=0.05)
    yolo_pred_df.to_csv(prediction_csv, index=False)
    print("Saved:", prediction_csv)

print("Predicted boxes:", len(yolo_pred_df))
display(yolo_pred_df.head())


In [ ]:

class PredictedCropDataset(Dataset):
    def __init__(self, dataframe, preprocess):
        self.df = dataframe.reset_index(drop=True)
        self.preprocess = preprocess

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        with Image.open(row.image_path) as im:
            im = im.convert("RGB")
            width, height = im.size

            x1 = max(0, int(math.floor(row.x1)))
            y1 = max(0, int(math.floor(row.y1)))
            x2 = min(width, int(math.ceil(row.x2)))
            y2 = min(height, int(math.ceil(row.y2)))

            if x2 <= x1 or y2 <= y1:
                crop = im
            else:
                crop = im.crop((x1, y1, x2, y2))

        return self.preprocess(crop), index

@torch.no_grad()
def encode_prediction_crops(dataframe, cache_path):
    cache_path = Path(cache_path)
    if cache_path.exists():
        obj = torch.load(cache_path, map_location="cpu")
        if len(obj["embeddings"]) == len(dataframe):
            print("Loaded:", cache_path)
            return obj["embeddings"]

    loader = DataLoader(
        PredictedCropDataset(dataframe, clip_preprocess),
        batch_size=CLIP_BATCH,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    all_embeddings = []
    for images, _ in loader:
        images = images.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            emb = clip_model.encode_image(images)
        all_embeddings.append(F.normalize(emb.float(), dim=-1).cpu())

    embeddings = torch.cat(all_embeddings)
    torch.save({"embeddings": embeddings}, cache_path)
    return embeddings

pred_crop_embeddings = encode_prediction_crops(
    yolo_pred_df,
    OUT_DIR / "yolo_prediction_crop_embeddings.pt",
)

semantic_pred, semantic_conf, _ = classify_embeddings(
    pred_crop_embeddings,
    best_fused_prototypes.cpu(),
    temperature=0.03,
)

adapter.eval()
with torch.no_grad():
    adapted_pred_embeddings = []
    for start in range(0, len(pred_crop_embeddings), 512):
        adapted_pred_embeddings.append(
            adapter(pred_crop_embeddings[start:start+512].to(DEVICE)).cpu()
        )
    adapted_pred_embeddings = torch.cat(adapted_pred_embeddings)

adapter_box_pred, adapter_box_conf, _ = classify_embeddings(
    adapted_pred_embeddings,
    best_fused_prototypes.cpu(),
    temperature=0.03,
)

yolo_pred_df["semantic_class"] = semantic_pred.numpy()
yolo_pred_df["semantic_conf"] = semantic_conf.numpy()
yolo_pred_df["adapter_class"] = adapter_box_pred.numpy()
yolo_pred_df["adapter_conf"] = adapter_box_conf.numpy()
yolo_pred_df.to_csv(OUT_DIR / "yolo_predictions_with_semantic_classes.csv", index=False)

display(yolo_pred_df.head())


## 13. Match YOLO boxes to ground truth and compare class correctness

In [ ]:

def xyxy_iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), dtype=np.float32)

    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])

    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = np.clip(a[:, 2] - a[:, 0], 0, None) * np.clip(a[:, 3] - a[:, 1], 0, None)
    area_b = np.clip(b[:, 2] - b[:, 0], 0, None) * np.clip(b[:, 3] - b[:, 1], 0, None)

    union = area_a[:, None] + area_b[None, :] - inter
    return inter / np.clip(union, 1e-9, None)

gt_test = records["test_set"].copy()
match_rows = []

for stem in sorted(set(gt_test.stem) | set(yolo_pred_df.stem)):
    gt_part = gt_test[gt_test.stem == stem].reset_index(drop=True)
    pred_part = yolo_pred_df[yolo_pred_df.stem == stem].reset_index(drop=True)

    if len(gt_part) == 0 or len(pred_part) == 0:
        continue

    gt_boxes = gt_part[["x1", "y1", "x2", "y2"]].to_numpy(float)
    pred_boxes = pred_part[["x1", "y1", "x2", "y2"]].to_numpy(float)
    ious = xyxy_iou_matrix(gt_boxes, pred_boxes)

    candidates = []
    for gi in range(len(gt_part)):
        for pi in range(len(pred_part)):
            if ious[gi, pi] >= 0.5:
                candidates.append((ious[gi, pi], gi, pi))
    candidates.sort(reverse=True)

    used_gt, used_pred = set(), set()
    for iou_value, gi, pi in candidates:
        if gi in used_gt or pi in used_pred:
            continue
        used_gt.add(gi)
        used_pred.add(pi)

        g = gt_part.iloc[gi]
        p = pred_part.iloc[pi]

        match_rows.append({
            "stem": stem,
            "iou": iou_value,
            "true_class": int(g.class_id),
            "yolo_class": int(p.yolo_class),
            "semantic_class": int(p.semantic_class),
            "adapter_class": int(p.adapter_class),
            "yolo_conf": float(p.yolo_conf),
            "semantic_conf": float(p.semantic_conf),
            "adapter_conf": float(p.adapter_conf),
            "aspect_ratio": float(g.aspect_ratio),
        })

matched_df = pd.DataFrame(match_rows)
print("Matched detections:", len(matched_df))

comparison_rows = []
for column, name in [
    ("yolo_class", "Original YOLO class"),
    ("semantic_class", "Semantic prototype class"),
    ("adapter_class", "Semantic adapter class"),
]:
    comparison_rows.append({
        "method": name,
        "matched_box_class_accuracy": accuracy_score(
            matched_df.true_class,
            matched_df[column],
        ),
        "matched_box_balanced_accuracy": balanced_accuracy_score(
            matched_df.true_class,
            matched_df[column],
        ),
    })

matched_comparison_df = pd.DataFrame(comparison_rows)
display(matched_comparison_df.round(4))

matched_df.to_csv(OUT_DIR / "matched_detection_classification.csv", index=False)
matched_comparison_df.to_csv(
    OUT_DIR / "matched_detection_classification_summary.csv",
    index=False,
)


In [ ]:

for column, title in [
    ("yolo_class", "Original YOLO classification on matched boxes"),
    ("semantic_class", "Semantic prototype reclassification on matched boxes"),
    ("adapter_class", "Semantic adapter reclassification on matched boxes"),
]:
    plot_confusion(
        matched_df.true_class.to_numpy(),
        matched_df[column].to_numpy(),
        title,
    )



## 14. Thin-crack diagnostic

Semantic embeddings primarily affect classification. If performance still collapses for high-aspect-ratio boxes, the remaining problem is likely localization or feature resolution rather than class semantics.


In [ ]:

matched_df["ar_bucket"] = pd.cut(
    matched_df.aspect_ratio,
    bins=[0, 2, 4, 8, np.inf],
    labels=["AR<2", "AR2-4", "AR4-8", "AR>=8"],
)

ar_rows = []
for bucket, part in matched_df.groupby("ar_bucket", observed=True):
    if len(part) == 0:
        continue
    ar_rows.append({
        "ar_bucket": str(bucket),
        "n": len(part),
        "yolo_class_accuracy": (part.yolo_class == part.true_class).mean(),
        "semantic_class_accuracy": (part.semantic_class == part.true_class).mean(),
        "adapter_class_accuracy": (part.adapter_class == part.true_class).mean(),
    })

ar_df = pd.DataFrame(ar_rows)
display(ar_df.round(4))
ar_df.to_csv(OUT_DIR / "matched_class_accuracy_by_aspect_ratio.csv", index=False)



## 15. Decision rule for the next research step

Use the results as follows:

### Continue toward a YOLO26 semantic head when

- enriched text beats class-name prompts;
- fused text + visual prototypes beat visual-only prototypes;
- semantic reclassification improves matched-box class accuracy;
- the biggest gains occur in known confusion pairs.

### Do not prioritize semantic-head integration when

- raw and adapted semantic classifiers are near random;
- visual prototypes strongly outperform all text-containing variants;
- most errors come from missed high-aspect-ratio objects rather than wrong labels.

In that case, prioritize P2 features, segmentation supervision, thin-object assignment, or geometry-aware losses.

## Main output files

- `official_yolo_metrics.csv`
- `alpha_sweep_validation.csv`
- `semantic_crop_test_results.csv`
- `semantic_crop_classification_report.csv`
- `semantic_crop_confusion_matrix.csv`
- `semantic_adapter_test_result.csv`
- `matched_detection_classification_summary.csv`
- `matched_class_accuracy_by_aspect_ratio.csv`
